# Sanity test — pipeline on Gemma-2-2B

Run this **before** paying for A100 hours on Gemma-3-12B. Validates the entire FaithEval pipeline end-to-end on a tiny subset, on a T4 (free Colab tier).

Expected runtime: ~10 min on T4. Cost: $0.

If this works, the M0.0 notebook is the same code with `variant="primary"` and no `limit`.

## Cell 1 — setup

On Colab: set HF_TOKEN and ANTHROPIC_API_KEY in Colab Secrets first, then:

In [ ]:
# uncomment on Colab
# from google.colab import userdata
# import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# !git clone https://github.com/<user>/desperation-circuit.git
# %cd desperation-circuit
!bash scripts/setup_env.sh

## Cell 2 — load Gemma-2-2B (sanity variant)

In [ ]:
import sys
sys.path.insert(0, '.')

from src.lib.model_load import load_gemma

model, tokenizer = load_gemma(variant='sanity')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}')

## Cell 3 — run 20 FaithEval prompts through the pipeline

In [ ]:
from pathlib import Path
from src.faitheval_eval import run_eval, summary

Path('outputs/sanity').mkdir(parents=True, exist_ok=True)
df = run_eval(
    model,
    tokenizer,
    limit=20,
    checkpoint_path=Path('outputs/sanity/faitheval_smoke.csv'),
    checkpoint_every=10,
)
df.head()

In [ ]:
summary(df)

## What this validates

- HF login works and Gemma weights download
- Tokenization + greedy decode runs without OOM on T4
- FaithEval dataset loads from HF
- Rule-based classifier returns sensible labels
- Claude judge gets called on ambiguous cases and parses correctly
- Checkpoint CSV writes

## If this works

Open `m0_viability.ipynb`. Same code, `variant='primary'`, A100 box, no `limit`. ~5 GPU-hr.

## If this fails

Fix here before paying for A100 hours. Common failures:
- Gemma license not accepted on HF → 403 on download
- ANTHROPIC_API_KEY missing → judge step crashes (ok to skip on this smoke test)
- T4 OOM → reduce `max_new_tokens` in `_greedy_generate`